<a href="https://colab.research.google.com/github/subbareddypullalacheruvu/Major-Project-Robust-Fraud-Detection-with-Spatial-Temporal-Features-and-Class-Imbalance-Learning-/blob/main/notebooks/tabformer_validation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Validation on the IBM TabFormer dataset

In `external_dataset_check.ipynb` we dropped the Financial Transactions dataset because its labels carry no signal, and we found that our six features miss most of the signal in the Credit Card Fraud 2026 dataset, where fraud does not come in bursts. Trilok suggested the IBM TabFormer credit card dataset (`ealtman2019/credit-card-transactions`). It is a synthetic dataset from IBM with about 24 million transactions from 2,000 users between 1991 and 2020. It has a user and card number, the full date and time, the amount, the merchant category code (MCC), and a separate users file with birth year and gender, so we can build all six of our features.

In this notebook we do two things:
1. **Check the dataset** in the same way as `external_dataset_check.ipynb`: overview, fraud rate, bursts and time gaps, and the score of each raw column on its own.
2. **Validate our final model** with the same protocol as on Sparkov: time-based split with the last 20% of the training period as validation, `category_risk` fitted on the training part only, Random Forest Tuned with five seeds, F2 threshold chosen on validation, and 95% day-block bootstrap intervals. We compare:
   - **our six features** (`amt_log`, `category_risk`, `is_night`, `gap_log`, `age`, `gender_flag`);
   - **our features without `gap_log`**, to see whether the time gap helps on this dataset;
   - **every raw column**, to see how much signal the dataset offers in total.

The full dataset is too large for Colab, so we take a random sample of users and keep every transaction of each chosen user, so the time gaps stay correct. We also keep only the most recent years for the experiments, but we calculate the time gaps on each card's full history first.

## 1. Environment setup
We type credentials in at runtime, so they are never saved in the notebook.

In [ ]:
# 1.1 Download the TabFormer dataset from Kaggle
import os
from getpass import getpass

os.environ['KAGGLE_USERNAME'] = input('Kaggle username: ')
os.environ['KAGGLE_KEY'] = getpass('Kaggle API key: ')

!kaggle datasets download -d ealtman2019/credit-card-transactions -p /content/data/tabformer --unzip

Kaggle username: Subbareddy_subbz
Kaggle API key: ··········
Dataset URL: https://www.kaggle.com/datasets/ealtman2019/credit-card-transactions
License(s): other
100% 263M/263M [00:12<00:00, 21.7MB/s]



In [ ]:
# 1.2 Clone the project repository
github_token = getpass('Paste your GitHub token: ').strip()
repo_url = f'https://{github_token}@github.com/subbareddypullalacheruvu/Major-Project-Robust-Fraud-Detection-with-Spatial-Temporal-Features-and-Class-Imbalance-Learning-.git'
!git clone {repo_url}
%cd Major-Project-Robust-Fraud-Detection-with-Spatial-Temporal-Features-and-Class-Imbalance-Learning-

Paste your GitHub token: ··········
Cloning into 'Major-Project-Robust-Fraud-Detection-with-Spatial-Temporal-Features-and-Class-Imbalance-Learning-'...
remote: Enumerating objects: 277, done.
remote: Counting objects: 100% (128/128), done.
remote: Compressing objects: 100% (86/86), done.
remote: Total 277 (delta 94), reused 42 (delta 42), pack-reused 149 (from 1)
Receiving objects: 100% (277/277), 11.67 MiB | 25.37 MiB/s, done.
Resolving deltas: 100% (172/172), done.
/content/Major-Project-Robust-Fraud-Detection-with-Spatial-Temporal-Features-and-Class-Imbalance-Learning-


In [ ]:
# 1.3 Imports and project helpers
import sys, glob
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import average_precision_score, roc_auc_score

sys.path.append(os.getcwd())
from src.utils import fit_category_risk, apply_category_risk
from src.evaluation import (chronological_val_split, run_seeds, bootstrap_model,
                            summarise, paired_difference, format_ci)
pd.set_option('display.max_columns', 60)

## 2. Settings and loading the data
### 2.1 Settings
- `N_USERS`: how many of the 2,000 users we sample. More users give more fraud cases but make training slower.
- `WINDOW_START`: the first date we use in the experiments. Time gaps are still calculated on the full history.
- `SEEDS`, `N_BOOT`, `BOOT_SEED`: the same as in our Sparkov notebooks.

We save each finished model and seed to Google Drive. The folder name includes the settings, so changing them never loads runs made with other settings.

In [ ]:
# 2.1 Settings and the Google Drive folder for saving progress
N_USERS      = 400
SAMPLE_SEED  = 42
WINDOW_START = '2015-01-01'
SEEDS        = [42, 43, 44, 45, 46]
N_BOOT       = 500
BOOT_SEED    = 2026

from google.colab import drive
drive.mount('/content/drive')
CHECKPOINT_DIR = f'/content/drive/MyDrive/fraud_checkpoints/tabformer_validation/u{N_USERS}_s{SAMPLE_SEED}_from{WINDOW_START[:4]}'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print('saving progress to ->', CHECKPOINT_DIR)

Mounted at /content/drive
saving progress to -> /content/drive/MyDrive/fraud_checkpoints/tabformer_validation/u400_s42_from2015


In [ ]:
# 2.2 List the downloaded files
DATA_DIR = '/content/data/tabformer'
files = sorted(f for f in glob.glob(f'{DATA_DIR}/**/*', recursive=True) if os.path.isfile(f))
for f in files:
    print(f'{os.path.getsize(f) / 1e6:9.1f} MB  {f}')

csvs = [f for f in files if f.endswith('.csv')]
users_csv = next(f for f in csvs if 'user' in os.path.basename(f).lower() and 'transaction' not in os.path.basename(f).lower())
txn_csv   = max((f for f in csvs if 'transaction' in os.path.basename(f).lower()), key=os.path.getsize)
print('\nusers file:', users_csv)
print('transactions file:', txn_csv)

      1.9 MB  /content/data/tabformer/User0_credit_card_transactions.csv
   2350.7 MB  /content/data/tabformer/credit_card_transactions-ibm_v2.csv
      0.5 MB  /content/data/tabformer/sd254_cards.csv
      0.2 MB  /content/data/tabformer/sd254_users.csv

users file: /content/data/tabformer/sd254_users.csv
transactions file: /content/data/tabformer/credit_card_transactions-ibm_v2.csv


In [ ]:
# 2.3 Load the users file and sample users
users = pd.read_csv(users_csv)
users['User'] = np.arange(len(users))          # the transactions refer to users by their row number
print(f'{len(users):,} users')
display(users.head(3))

rng = np.random.default_rng(SAMPLE_SEED)
sampled_users = set(rng.choice(users['User'].values, size=min(N_USERS, len(users)), replace=False).tolist())
print(f'sampled {len(sampled_users)} users')

2,000 users


,Person,Current Age,Retirement Age,Birth Year,Birth Month,Gender,Address,Apartment,City,State,Zipcode,Latitude,Longitude,Per Capita Income - Zipcode,Yearly Income - Person,Total Debt,FICO Score,Num Credit Cards,User
0,Hazel Robinson,53,66,1966,11,Female,462 Rose Lane,NaN,La Verne,CA,91750,34.15,-117.76,$29278,$59696,$127613,787,5,0
1,Sasha Sadr,53,68,1966,12,Female,3606 Federal Boulevard,NaN,Little Neck,NY,11363,40.76,-73.74,$37891,$77254,$191349,701,5,1
2,Saanvi Lee,81,67,1938,11,Female,766 Third Drive,NaN,West Covina,CA,91792,34.02,-117.89,$22681,$33483,$196,698,5,2


sampled 400 users


In [ ]:
# 2.4 Read the transactions in chunks and keep the sampled users
parts, total = [], 0
for chunk in pd.read_csv(txn_csv, chunksize=2_000_000, low_memory=False):
    total += len(chunk)
    parts.append(chunk[chunk['User'].isin(sampled_users)])
raw = pd.concat(parts, ignore_index=True)
del parts
print(f'read {total:,} transactions, kept {len(raw):,} from {raw["User"].nunique()} users')

read 24,386,900 transactions, kept 5,018,900 from 400 users


## 3. Overview

In [ ]:
# 3.1 Column overview of the transactions
def column_overview(df):
    return pd.DataFrame({
        'type': df.dtypes.astype(str),
        'distinct values': df.nunique(),
        'missing %': (df.isna().mean() * 100).round(2),
        'example': df.iloc[0],
    })

display(column_overview(raw))
display(raw.head(3))

,type,distinct values,missing %,example
User,int64,400,0.00,10
Card,int64,9,0.00,0
Year,int64,28,0.00,2020
Month,int64,12,0.00,2
Day,int64,31,0.00,2
Time,object,1440,0.00,03:26
Amount,object,56652,0.00,$45.57
Use Chip,object,3,0.00,Chip Transaction
Merchant Name,int64,37114,0.00,7479593622964634169
Merchant City,object,9901,0.00,Fair Oaks


,User,Card,Year,Month,Day,Time,Amount,Use Chip,Merchant Name,Merchant City,Merchant State,Zip,MCC,Errors?,Is Fraud?
0,10,0,2020,2,2,03:26,$45.57,Chip Transaction,7479593622964634169,Fair Oaks,CA,95628.0,5311,Insufficient Balance,No
1,10,0,2020,2,3,02:47,$53.75,Swipe Transaction,97032797689821735,Fair Oaks,CA,95628.0,5411,NaN,No
2,10,0,2020,2,3,09:31,$132.77,Chip Transaction,-2472481739355111587,Fair Oaks,CA,95628.0,7538,NaN,No


## 4. Preparing the columns
TabFormer stores the date in separate `Year`, `Month`, `Day` and `Time` columns, the amount as text (for example `$134.09`) and the label as `Yes`/`No`. We build one timestamp, a numeric amount and a 0/1 label. A card is identified by the user and the card number together.

Age and gender come from the users file. The age is the age at the time of the transaction, calculated from the birth year and month.

We calculate the time since the card's previous transaction on the full history, and only then keep the transactions from `WINDOW_START`. This way the first transactions in the window still get their real time gap.

In [ ]:
# 4.1 Timestamp, amount, label, card and the users' age and gender
def find(df, *names):
    lower = {c.lower(): c for c in df.columns}
    return next((lower[n.lower()] for n in names if n.lower() in lower), None)

LABEL_COL = find(raw, 'Is Fraud?', 'is_fraud')
d = raw.copy()
d['_time'] = pd.to_datetime(dict(year=d['Year'], month=d['Month'], day=d['Day']), errors='coerce') \
             + pd.to_timedelta(d['Time'].astype(str) + ':00', errors='coerce')
d['_y'] = d[LABEL_COL].astype(str).str.strip().str.lower().isin(['yes', '1', 'true']).astype(int)
d['amount'] = pd.to_numeric(d['Amount'].astype(str).str.replace(r'[$,]', '', regex=True), errors='coerce')
d['card_id'] = d['User'].astype(str) + '_' + d['Card'].astype(str)

BIRTH_YEAR, BIRTH_MONTH, GENDER = find(users, 'Birth Year'), find(users, 'Birth Month'), find(users, 'Gender')
d = d.merge(users[['User', BIRTH_YEAR, BIRTH_MONTH, GENDER]], on='User', how='left')
birth = pd.to_datetime(dict(year=d[BIRTH_YEAR], month=d[BIRTH_MONTH], day=1), errors='coerce')
d['age'] = (d['_time'] - birth).dt.days / 365.25
d['gender_flag'] = d[GENDER].astype(str).str.strip().str.upper().str.startswith('M').astype(int)

print('unparsed times:', d['_time'].isna().sum(), '| unparsed amounts:', d['amount'].isna().sum())
d = d.dropna(subset=['_time'])
print(f"{len(d):,} rows from {d['_time'].min()} to {d['_time'].max()}, fraud rate {d['_y'].mean():.4%}")

unparsed times: 0 | unparsed amounts: 0
5,018,900 rows from 1993-06-24 12:05:00 to 2020-02-28 23:49:00, fraud rate 0.1144%


In [ ]:
# 4.2 Time gap on the full history, then keep the window
d = d.sort_values(['card_id', '_time'])
d['_gap_h'] = d.groupby('card_id')['_time'].diff().dt.total_seconds() / 3600
d = d[d['_time'] >= WINDOW_START].sort_values('_time').reset_index(drop=True)
print(f"window from {WINDOW_START}: {len(d):,} rows, {d['_y'].sum():,} fraud ({d['_y'].mean():.4%})")
print(d.groupby(d['_time'].dt.year)['_y'].agg(['size', 'sum']).rename(columns={'size': 'rows', 'sum': 'fraud'}))

window from 2015-01-01: 1,841,549 rows, 2,182 fraud (0.1185%)
         rows  fraud
_time               
2015   351144    544
2016   353585    758
2017   356184     47
2018   355436    445
2019   356542    388
2020    68658      0


## 5. Fraud rate, cards, bursts and time gaps
We compare the same facts as for the other datasets. In Sparkov, fraud comes in bursts on one card within a day and fraud transactions have much shorter time gaps (median 1.33 h against 4.57 h). If TabFormer shows the same pattern, we expect `gap_log` to help.

In [ ]:
# 5.1 Basic facts and the time gaps
NIGHT_HOURS = [22, 23, 0, 1, 2, 3]
per_card = d.groupby('card_id').size()
f = d[d['_y'] == 1]
per_card_day = f.groupby(['card_id', f['_time'].dt.floor('D')]).size()
facts = pd.Series({
    'rows': len(d), 'fraud rows': int(d['_y'].sum()), 'fraud rate %': round(d['_y'].mean() * 100, 4),
    'start': d['_time'].min(), 'end': d['_time'].max(),
    'users': d['User'].nunique(), 'cards': per_card.size,
    'median txns per card': per_card.median(),
    'cards with any fraud': f['card_id'].nunique(),
    'median gap h, fraud': round(f['_gap_h'].median(), 2),
    'median gap h, normal': round(d.loc[d['_y'] == 0, '_gap_h'].median(), 2),
    'mean frauds per card-day': round(per_card_day.mean(), 2),
    'frauds in card-days with 2+ frauds %': round(per_card_day[per_card_day >= 2].sum() / per_card_day.sum() * 100, 1),
    'fraud at night (22-03) %': round(f['_time'].dt.hour.isin(NIGHT_HOURS).mean() * 100, 1),
    'normal at night (22-03) %': round(d.loc[d['_y'] == 0, '_time'].dt.hour.isin(NIGHT_HOURS).mean() * 100, 1),
})
facts

,0
rows,1841549
fraud rows,2182
fraud rate %,0.1185
start,2015-01-01 00:01:00
end,2020-02-28 23:49:00
users,396
cards,1107
median txns per card,1161.0
cards with any fraud,317
"median gap h, fraud",2.51


## 6. Splitting by time and building the features
The same split as on Sparkov: the last 20% of the period is the test set, and the last 20% of the remaining training period is validation.

Our features:
- `amt_log`: log of the amount (refunds have negative amounts, so we use 0 for them);
- `category_risk`: the fraud rate of each MCC on the training part only;
- `is_night`: 1 for transactions between 22:00 and 03:59;
- `gap_log`: log of the hours since the card's previous transaction, -1 for a card's first transaction;
- `age` and `gender_flag` from the users file.

TabFormer has merchant city and state but no merchant coordinates, so we cannot build `velocity_log` here.

In [ ]:
# 6.1 Split by time
cutoff_test = d['_time'].min() + 0.8 * (d['_time'].max() - d['_time'].min())
train = d[d['_time'] < cutoff_test]
test  = d[d['_time'] >= cutoff_test].copy()
train_sub, val, cutoff_val = chronological_val_split(train, '_time', val_frac=0.2)

print('validation starts:', cutoff_val, '| test starts:', cutoff_test)
for name, part in [('train_sub', train_sub), ('val', val), ('test', test)]:
    print(f"{name:9s} rows={len(part):>9,}  fraud={part['_y'].sum():>6,}  fraud rate={part['_y'].mean():.4%}")

validation starts: 2018-04-21 09:28:12 | test starts: 2019-02-16 23:51:24
train_sub rows=1,168,504  fraud= 1,511  fraud rate=0.1293%
val       rows=  293,511  fraud=   362  fraud rate=0.1233%
test      rows=  379,534  fraud=   309  fraud rate=0.0814%


In [ ]:
# 6.2 Build our features (category_risk fitted on the training part only)
MCC_COL = find(d, 'MCC')
risk_map = fit_category_risk(train_sub, MCC_COL, '_y')
for part in (train_sub, val, test):
    part['amt_log'] = np.log1p(part['amount'].clip(lower=0)).fillna(0)
    part['category_risk'] = apply_category_risk(part, MCC_COL, risk_map)
    part['is_night'] = part['_time'].dt.hour.isin(NIGHT_HOURS).astype(int)
    part['gap_log'] = np.log1p(part['_gap_h']).fillna(-1)
    part['age'] = part['age'].fillna(-1)
    part['day'] = part['_time'].dt.floor('D')

SIX    = ['amt_log', 'category_risk', 'is_night', 'gap_log', 'age', 'gender_flag']
NO_GAP = [c for c in SIX if c != 'gap_log']
y_tr, y_val, y_test = train_sub['_y'], val['_y'], test['_y']
test_days = test['day'].values

## 7. Each raw column on its own
Before training, we score every raw column of the transactions and users files on its own on the test set, in the same way as in `external_dataset_check.ipynb`:
- a **numeric column** is used directly as a score; we decide on the training part whether high or low values mean fraud;
- a **text column** with up to 1,000 values, and code columns such as the MCC and the state, are replaced by their fraud rate on the training part;
- identifiers (user, card, merchant name, address, zip codes) and columns with more than 1,000 text values are skipped.

A ROC-AUC of 0.95 or more means the column almost gives the answer, so we check it for leakage. Below 0.55 means little or no signal.

In [ ]:
# 7.1 Score every raw column on its own
SKIP = {'User', 'Card', 'Year', 'Month', 'Day', 'Time', 'Amount', LABEL_COL, 'Merchant Name', 'Zip',
        'Person', 'Address', 'Apartment', 'Zipcode', 'Birth Month', 'card_id'}
user_cols = [c for c in users.columns if c not in SKIP and c not in raw.columns]
for part in (train_sub, val, test):
    extra = part[['User']].merge(users[['User'] + user_cols], on='User', how='left')
    for c in user_cols:
        if c not in part.columns:
            part[c] = extra[c].values
RAW_COLS = [c for c in list(raw.columns) + user_cols if c not in SKIP] + ['amount']
RAW_COLS = list(dict.fromkeys(RAW_COLS))

AS_TEXT = {MCC_COL, 'Merchant State', 'State', 'Gender'}   # codes and names, not amounts

def encode(col):
    # numeric: fill with the training median; text: fraud rate of each value on the training part
    if col in AS_TEXT:
        if train_sub[col].nunique() > 1000:
            return None, None
        rates = y_tr.groupby(train_sub[col].astype(str)).mean()
        return 'fraud rate of each value', (lambda p: p[col].astype(str).map(rates).fillna(y_tr.mean()))
    if pd.api.types.is_numeric_dtype(train_sub[col]):
        s = pd.to_numeric(train_sub[col].astype(str).str.replace(r'[$,]', '', regex=True), errors='coerce')
        fill = s.median()
        enc = lambda p: pd.to_numeric(p[col].astype(str).str.replace(r'[$,]', '', regex=True), errors='coerce').fillna(fill)
        return 'numeric', enc
    as_num = pd.to_numeric(train_sub[col].astype(str).str.replace(r'[$,]', '', regex=True), errors='coerce')
    if as_num.notna().mean() > 0.99:                      # numbers stored as text, such as '$59,696'
        fill = as_num.median()
        enc = lambda p: pd.to_numeric(p[col].astype(str).str.replace(r'[$,]', '', regex=True), errors='coerce').fillna(fill)
        return 'numeric', enc
    if train_sub[col].nunique() > 1000:
        return None, None
    rates = y_tr.groupby(train_sub[col].astype(str)).mean()
    enc = lambda p: p[col].astype(str).map(rates).fillna(y_tr.mean())
    return 'fraud rate of each value', enc

column_rows, encoders = [], {}
for col in RAW_COLS:
    used, enc = encode(col)
    if enc is None:
        column_rows.append({'column': col, 'used as': 'skipped (too many text values)'}); continue
    x_tr, x_te = enc(train_sub), enc(test)
    sign = -1 if (used == 'numeric' and x_tr.nunique() > 1 and roc_auc_score(y_tr, x_tr) < 0.5) else 1
    encoders[col] = enc
    if x_te.nunique() < 2:
        column_rows.append({'column': col, 'used as': used + ' (constant on test)'}); continue
    pr, roc = average_precision_score(y_test, sign * x_te), roc_auc_score(y_test, sign * x_te)
    column_rows.append({'column': col, 'used as': used, 'PR-AUC': round(pr, 4),
                        'lift': round(pr / y_test.mean(), 2), 'ROC-AUC': round(roc, 4)})

column_df = pd.DataFrame(column_rows)
column_df['note'] = np.select([column_df['ROC-AUC'] >= 0.95, column_df['ROC-AUC'] < 0.55],
                              ['check for leakage', 'little or no signal'], 'some signal')
column_df.loc[column_df['ROC-AUC'].isna(), 'note'] = ''
column_df = column_df.sort_values('ROC-AUC', ascending=False, na_position='last').reset_index(drop=True)
print(f'test fraud rate {y_test.mean():.4%}')
column_df

test fraud rate 0.0814%


,column,used as,PR-AUC,lift,ROC-AUC,note
0,Merchant State,fraud rate of each value,0.7843,963.28,0.9999,check for leakage
1,MCC,fraud rate of each value,0.0108,13.29,0.7952,some signal
2,City,fraud rate of each value,0.0030,3.70,0.6266,some signal
3,Current Age,numeric,0.0012,1.46,0.5878,some signal
4,Birth Year,numeric,0.0012,1.45,0.5877,some signal
5,State,fraud rate of each value,0.0014,1.66,0.5592,some signal
6,amount,numeric,0.0019,2.32,0.5559,some signal
7,Num Credit Cards,numeric,0.0010,1.27,0.5289,little or no signal
8,Total Debt,numeric,0.0009,1.16,0.5270,little or no signal
9,Retirement Age,numeric,0.0008,1.04,0.5200,little or no signal


In [ ]:
# 7.2 Our features on their own, for comparison
our_rows = []
for col in SIX:
    x_tr, x_te = train_sub[col], test[col]
    sign = -1 if roc_auc_score(y_tr, x_tr) < 0.5 else 1
    pr, roc = average_precision_score(y_test, sign * x_te), roc_auc_score(y_test, sign * x_te)
    our_rows.append({'feature': col, 'PR-AUC': round(pr, 4), 'lift': round(pr / y_test.mean(), 2), 'ROC-AUC': round(roc, 4)})
our_feature_df = pd.DataFrame(our_rows).sort_values('ROC-AUC', ascending=False).reset_index(drop=True)
our_feature_df

,feature,PR-AUC,lift,ROC-AUC
0,category_risk,0.0108,13.29,0.7952
1,age,0.0011,1.37,0.5872
2,amt_log,0.0019,2.32,0.5558
3,is_night,0.0009,1.08,0.5386
4,gap_log,0.0008,1.01,0.5257
5,gender_flag,0.0008,1.00,0.5001


## 8. Validation with our protocol
### 8.1 Training
We train the three runs with the Random Forest Tuned settings and five seeds each. For the "every raw column" run we use every column from Section 7 that was not skipped, encoded with the training part only. This run is only there to measure how much signal the dataset has; the ready-made columns are not part of our method.

This is the longest step in the notebook. Finished seeds are saved to Google Drive, so if Colab disconnects, re-running skips them.

In [20]:
# 8.1 Train the three runs with every seed
RAW_FEATURES = []
for col in encoders:
    name = f'raw__{col}'
    for part in (train_sub, val, test):
        part[name] = encoders[col](part).values
    RAW_FEATURES.append(name)
print(f'every raw column run uses {len(RAW_FEATURES)} columns')

def make_rf(n_features):
    return lambda s: RandomForestClassifier(n_estimators=200, max_features=min(3, n_features),
                                            min_samples_leaf=5, random_state=s, n_jobs=-1)

# location features: is the merchant outside the cardholder's home state, or abroad?
for part in (train_sub, val, test):
    ms  = part['Merchant State']
    has = ms.notna()
    part['abroad'] = (has & (ms.astype(str).str.len() > 2)).astype(int)
    part['outside_home_state'] = (has & (ms.astype(str) != part['State'].astype(str))).astype(int)

RUNS = {'Our six features': SIX,
        'Without gap_log': NO_GAP,
        'Every raw column': RAW_FEATURES,
        'Our six + location': SIX + ['outside_home_state', 'abroad'],
        'Every raw column without Merchant State': [c for c in RAW_FEATURES if c != 'raw__Merchant State']}
runs = {}
for name, cols in RUNS.items():
    runs[name] = run_seeds(name, make_rf(len(cols)),
                           train_sub[cols], y_tr, val[cols], y_val, test[cols], SEEDS,
                           checkpoint_dir=CHECKPOINT_DIR)

every raw column run uses 18 columns
Our six features seed 42: loaded from checkpoint
Our six features seed 43: loaded from checkpoint
Our six features seed 44: loaded from checkpoint
Our six features seed 45: loaded from checkpoint
Our six features seed 46: loaded from checkpoint
Without gap_log seed 42: loaded from checkpoint
Without gap_log seed 43: loaded from checkpoint
Without gap_log seed 44: loaded from checkpoint
Without gap_log seed 45: loaded from checkpoint
Without gap_log seed 46: loaded from checkpoint
Every raw column seed 42: loaded from checkpoint
Every raw column seed 43: loaded from checkpoint
Every raw column seed 44: loaded from checkpoint
Every raw column seed 45: loaded from checkpoint
Every raw column seed 46: loaded from checkpoint
Our six + location seed 42: threshold=0.09  val PR-AUC=0.3913
Our six + location seed 43: threshold=0.08  val PR-AUC=0.3969
Our six + location seed 44: threshold=0.08  val PR-AUC=0.3893
Our six + location seed 45: threshold=0.09  val

### 8.2 Results table
500 day-block resamples, the same `BOOT_SEED` for every run, seed average with a 95% interval, as on Sparkov.

In [21]:
# 8.2 Bootstrap each run and build the results table
boots, summaries = {}, {}
for name, seed_runs in runs.items():
    pairs = [(r['test_proba'], r['threshold']) for r in seed_runs]
    boots[name] = bootstrap_model(y_test, pairs, test_days, n_boot=N_BOOT, boot_seed=BOOT_SEED)
    summaries[name] = summarise(y_test, pairs, boots[name])
    summaries[name]['Val PR-AUC'] = np.mean([r['val_pr_auc'] for r in seed_runs])
    print(f'{name}: done')

summary_df = pd.DataFrame(summaries).T
table = pd.DataFrame({
    'Val PR-AUC': summary_df['Val PR-AUC'].astype(float).round(4),
    'PR-AUC [95% CI]': [format_ci(r, 'PR-AUC') for _, r in summary_df.iterrows()],
    'F1 [95% CI]':  [format_ci(r, 'F1')  for _, r in summary_df.iterrows()],
    'F2 [95% CI]':  [format_ci(r, 'F2')  for _, r in summary_df.iterrows()],
    'MCC [95% CI]': [format_ci(r, 'MCC') for _, r in summary_df.iterrows()],
    'Thresholds (per seed)': summary_df['Thresholds'],
})
print(f'test fraud rate {y_test.mean():.4%}  ({int(y_test.sum()):,} fraud transactions)')
table

Our six features: done
Without gap_log: done
Every raw column: done
Our six + location: done
Every raw column without Merchant State: done
test fraud rate 0.0814%  (309 fraud transactions)


,Val PR-AUC,PR-AUC [95% CI],F1 [95% CI],F2 [95% CI],MCC [95% CI],Thresholds (per seed)
Our six features,0.0375,"0.0182 [0.0085, 0.0335]","0.0403 [0.0287, 0.0533]","0.0700 [0.0505, 0.0892]","0.0554 [0.0403, 0.0704]","0.04, 0.04, 0.04, 0.05, 0.05"
Without gap_log,0.0459,"0.0288 [0.0142, 0.0473]","0.0450 [0.0325, 0.0585]","0.0770 [0.0576, 0.0971]","0.0608 [0.0449, 0.0773]","0.05, 0.05, 0.06, 0.06, 0.05"
Every raw column,0.8875,"0.8914 [0.8558, 0.9281]","0.7742 [0.7241, 0.8239]","0.8405 [0.7973, 0.8803]","0.7812 [0.7322, 0.8285]","0.25, 0.21, 0.21, 0.24, 0.20"
Our six + location,0.3921,"0.3456 [0.2831, 0.4070]","0.2954 [0.2542, 0.3428]","0.4125 [0.3665, 0.4546]","0.3350 [0.2952, 0.3744]","0.09, 0.08, 0.08, 0.09, 0.10"
Every raw column without Merchant State,0.0286,"0.0033 [0.0024, 0.0049]","0.0173 [0.0099, 0.0256]","0.0296 [0.0175, 0.0426]","0.0222 [0.0126, 0.0325]","0.05, 0.04, 0.05, 0.05, 0.06"


### 8.3 Paired differences
If the 95% interval includes zero, we cannot tell the two runs apart.

In [22]:
# 8.3 Paired differences (first run minus second run)
COMPARISONS = [('Our six features', 'Without gap_log'),
               ('Every raw column', 'Our six features'),
               ('Our six + location', 'Our six features'),
               ('Every raw column', 'Every raw column without Merchant State')]
diff_rows = []
for a, b in COMPARISONS:
    for metric in ['PR-AUC', 'F1', 'F2', 'MCC']:
        dd = paired_difference(boots[a], boots[b], metric)
        diff_rows.append({'Comparison': f'{a} minus {b}', 'Metric': metric,
                          **{k: round(v, 4) for k, v in dd.items()}})
diff_df = pd.DataFrame(diff_rows).rename(columns={'P(A > B)': 'P(first > second)'})
diff_df

,Comparison,Metric,mean diff,CI low,CI high,P(first > second)
0,Our six features minus Without gap_log,PR-AUC,-0.0105,-0.0215,-0.0014,0.016
1,Our six features minus Without gap_log,F1,-0.0048,-0.0123,0.0034,0.132
2,Our six features minus Without gap_log,F2,-0.0072,-0.0202,0.0071,0.166
3,Our six features minus Without gap_log,MCC,-0.0055,-0.0162,0.0063,0.176
4,Every raw column minus Our six features,PR-AUC,0.8732,0.8341,0.9085,1.000
5,Every raw column minus Our six features,F1,0.7338,0.6864,0.7773,1.000
6,Every raw column minus Our six features,F2,0.7707,0.7290,0.8105,1.000
7,Every raw column minus Our six features,MCC,0.7259,0.6799,0.7690,1.000
8,Our six + location minus Our six features,PR-AUC,0.3287,0.2690,0.3820,1.000
9,Our six + location minus Our six features,F1,0.2561,0.2193,0.2952,1.000


## 9. Saving the results

In [23]:
# 9.1 Save the results tables
os.makedirs('results', exist_ok=True)
facts.to_csv('results/tabformer_facts.csv')
column_df.to_csv('results/tabformer_columns.csv', index=False)
our_feature_df.to_csv('results/tabformer_our_features.csv', index=False)
summary_df.to_csv('results/tabformer_summary.csv')
table.to_csv('results/tabformer_table.csv')
diff_df.to_csv('results/tabformer_paired_differences.csv', index=False)
print('saved to results/')

saved to results/


## 10. Discussion